<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m3_01_manual_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M3.01 — Tuning a spam filter by hand

In the lab we read and changed the hyperparameters of a pipeline with
`get_params` and `set_params`, and searched for good values with our own
loops. In this workbook you tune **a spam filter**. You will count what a
search costs before running it, see why the values to try are chosen on a
logarithmic scale, learn to recognise a grid that is too narrow, and
treat a preprocessing step as one more hyperparameter.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "best_position": "a1cb100f57e971ca",
    "better_first_step": "f9a5f16df9dda930",
    "extend_towards": "4f627c9e32f4ff3e",
    "n_fits": "9f14025af0065b30",
    "n_fits_two_parameters": "ff5a1ae012afa5d4",
    "n_linear_below_100": "6b86b273ff34fce1",
    "param_name": "0a0c82b6ff2928f9",
    "worst_C": "8aed642bf5118b9d"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

Each row describes one e-mail received at a research laboratory in the
1990s. Nobody kept the text: for each e-mail we only have **counts**.

- `word_freq_…`: how often a word appears, as a percentage of the words
  of the e-mail (`word_freq_free`, `word_freq_money`, `word_freq_george`…);
- `char_freq_…`: the same for a few characters (`!`, `$`…);
- `capital_run_length_…`: statistics on sequences of CAPITAL LETTERS.

`is_spam` is 1 for unwanted e-mails and 0 for normal ones.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

spam = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/spambase.csv"
)
data = spam.drop(columns="is_spam")
target = spam["is_spam"]
print(f"{data.shape[0]} e-mails, {data.shape[1]} features, share of spam: {target.mean():.2f}")
spam[["word_freq_free", "word_freq_george", "char_freq_exclamation", "capital_run_length_total", "is_spam"]].head()

The file lists all the spam first, then all the normal e-mails. As in
workbook M1.03 we therefore **shuffle** the rows when cutting them into
folds, and we use the same five folds for every comparison below.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def accuracy(model):
    """Mean and standard deviation of the accuracy over the five folds."""
    scores = cross_validate(model, data, target, cv=cv)["test_score"]
    return scores.mean(), scores.std()

## 2. Reading the hyperparameters of a pipeline

Our spam filter scales the features and feeds them to a logistic
regression.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
mean, std = accuracy(model)
print(f"accuracy with default settings: {mean:.3f} ± {std:.3f}")

A **hyperparameter** is a setting we choose before training, as opposed
to the weights the model learns by itself. `get_params()` lists them all.
Inside a pipeline, each name is built as
`<name of the step>__<name of the parameter>`, with **two** underscores:

In [ ]:
[name for name in model.get_params() if "__" in name]

The logistic regression has a hyperparameter `C` that controls how freely
it may choose its weights: a small `C` forces small weights (a cautious
model), a large `C` lets them grow.

✏️ Write the full name of this hyperparameter **inside our pipeline** (a
string, copied from the list above).

In [ ]:
param_name = ____
check("param_name", param_name, hint="Step name, two underscores, parameter name.")
print("current value:", model.get_params()[param_name])

## 3. A search by hand

`set_params` changes a hyperparameter. To find a good `C` we try several
values and cross-validate the pipeline for each of them.

✏️ **Before running the loop**, count what it costs. We try 7 values of
`C`, each evaluated on 5 folds. How many times is the pipeline trained?

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
n_fits = ____
check("n_fits", n_fits, hint="One training per value and per fold.")

In [ ]:
rows = []
for C in C_values:
    model.set_params(**{param_name: C})
    mean, std = accuracy(model)
    rows.append({"C": C, "mean accuracy": mean, "std": std})
by_hand = pd.DataFrame(rows).set_index("C")
by_hand.round(3)

✏️ Which of the 7 values of `C` gives the **worst** spam filter?

In [ ]:
worst_C = ____
check("worst_C", worst_C)

🧠 **Think.** Look at the values of `C` from 0.1 upwards. Compare the
differences between their mean accuracies with the `std` column. Can you
say which of them is "the best"? Does it matter?

### ➕ Going further: why 0.001, 0.01, 0.1… and not 1, 2, 3…?

Our 7 values are spread **on a logarithmic scale**: each one is 10 times
the previous one. Suppose we had instead asked for 7 values spread
**evenly** between the same smallest and largest value:

In [ ]:
linear_values = np.linspace(0.001, 1000, 7)
[round(float(value), 1) for value in linear_values]

✏️ How many of these 7 evenly spread values are **below 100**?

In [ ]:
n_linear_below_100 = ____
check("n_linear_below_100", n_linear_below_100)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 1.8))
ax.plot(C_values, [1] * 7, "o", label="logarithmic grid")
ax.plot(linear_values, [0] * 7, "s", label="evenly spread grid")
ax.set(xscale="log", yticks=[], ylim=(-0.6, 1.6), xlabel="value of C (log scale)")
_ = ax.legend(loc="center left")

The table of section 3 shows that everything interesting happens between
0.001 and 1: that is where the accuracy changes. The evenly spread grid
has a **single** point there and spends the six others in a region where
nothing moves. For a hyperparameter whose useful values cover several
orders of magnitude, a logarithmic grid looks everywhere.

### ➕ Going further: a grid that is too narrow

A colleague is afraid of large weights and searches only among small
values of `C`:

In [ ]:
narrow_values = [0.0001, 0.001, 0.01]
rows = []
for C in narrow_values:
    model.set_params(**{param_name: C})
    mean, std = accuracy(model)
    rows.append({"C": C, "mean accuracy": mean, "std": std})
narrow = pd.DataFrame(rows).set_index("C")
narrow.round(3)

✏️ Where is the best value of this search: in the `"middle"` of the grid
or on its `"edge"`? And in which direction should the colleague extend
the grid: towards `"smaller"` or `"larger"` values of `C`?

In [ ]:
best_position = ____
extend_towards = ____
check("best_position", best_position)
check("extend_towards", extend_towards)

When the best value sits on the edge of the grid, the search has not
found an optimum: it has found the **limit of what you allowed it to
try**. The score may keep improving beyond it. Always look at where the
winner is.

## 4. ➕ Going further: preprocessing is a hyperparameter too

Look at one feature, the total number of capital letters in an e-mail:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(data["capital_run_length_total"], bins=50)
axes[0].set(title="capital_run_length_total", xlabel="value", ylabel="number of e-mails")
axes[1].hist(np.log1p(data["capital_run_length_total"]), bins=50)
_ = axes[1].set(title="log(1 + capital_run_length_total)", xlabel="value")

On the left, almost all e-mails are squeezed into the first bar, and a
few e-mails written entirely in capitals are far to the right. Most of
our 57 features look like this. A linear model multiplies each feature by
one weight, so these few extreme e-mails pull hard on it.

A logarithm compresses large values and spreads small ones (right). We
add it as a first step of the pipeline. `np.log1p(x)` computes
`log(1 + x)`, which is 0 when the count is 0.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

model = make_pipeline(
    FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=1000)
)
list(model.named_steps)

Is the logarithm a good idea? That is a question for cross-validation,
exactly like the value of `C`. A whole step is a parameter of the
pipeline too: setting it to the string `"passthrough"` switches it off.

✏️ We now try the 7 values of `C` **with and without** the logarithm,
still on 5 folds. How many times is the pipeline trained?

In [ ]:
n_fits_two_parameters = ____
check("n_fits_two_parameters", n_fits_two_parameters, hint="Every combination of the two settings is evaluated on every fold.")

🔮 **Predict first.** Will the logarithm make the spam filter
**"worse"**, **"about the same"** or **"better"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["worse", "about the same", "better"])

In [ ]:
first_steps = {"with log": FunctionTransformer(np.log1p), "without log": "passthrough"}
rows = []
for label, step in first_steps.items():
    for C in C_values:
        model.set_params(functiontransformer=step, logisticregression__C=C)
        mean, std = accuracy(model)
        rows.append({"first step": label, "C": C, "mean accuracy": mean})
two_parameters = pd.DataFrame(rows).pivot(index="C", columns="first step", values="mean accuracy")
two_parameters.round(3)

✏️ Which first step gives the better spam filter? Write `"with log"` or
`"without log"`.

In [ ]:
better_first_step = ____
check("better_first_step", better_first_step)

🧠 **Think.**

1. An accuracy of 0.94 against 0.93 looks like a small gain. Out of 1000
   e-mails, how many **errors** does each filter make? By what share did
   the logarithm cut the errors?
2. Adding one setting with 2 options doubled the number of trainings. A
   third setting with 5 options would multiply it again by 5. What does
   that tell you about searching many hyperparameters by hand?
3. The best `C` is not the same with and without the logarithm. Why is it
   risky to tune one hyperparameter, fix it, and then tune the next?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - how parameter names are built in a pipeline and how you change one;
   - why the values of `C` are spread on a logarithmic scale;
   - how to recognise a grid that is too narrow, and what to do;
   - why a preprocessing step can be tuned like any other setting, and
     what the logarithm changed here.